#### STEP 1: Load libraries and data

In [76]:
import pandas as pd
import numpy as np

In [77]:
# Load data from the processed CSV file
df = pd.read_csv("../data/processed/loans_cleaned.csv")
economic_df = pd.read_csv("../data/raw/economic_indicators.csv")

df["disbursement_date"] = pd.to_datetime(df["disbursement_date"])
df["due_date"] = pd.to_datetime(df["due_date"])

In [78]:
def check_df(df):
    print("Dataframe Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Data Types:\n", df.dtypes)

check_df(df)

Dataframe Shape: (68654, 16)
Columns: ['id', 'customer_id', 'country_id', 'tbl_loan_id', 'lender_id', 'loan_type', 'total_amount', 'total_amount_to_repay', 'disbursement_date', 'due_date', 'duration', 'new_versus_repeat', 'amount_funded_by_lender', 'lender_portion_funded', 'lender_portion_to_be_repaid', 'target']
Data Types:
 id                                        str
customer_id                             int64
country_id                                str
tbl_loan_id                             int64
lender_id                               int64
loan_type                                 str
total_amount                          float64
total_amount_to_repay                 float64
disbursement_date              datetime64[us]
due_date                       datetime64[us]
duration                                int64
new_versus_repeat                         str
amount_funded_by_lender               float64
lender_portion_funded                 float64
lender_portion_to_be_repaid 

##### 1. Date-Based Features

In [79]:
df["loan_year"] = df["disbursement_date"].dt.year
df["loan_month"] = df["disbursement_date"].dt.month
df["loan_day_of_week"] = df["disbursement_date"].dt.dayofweek

In [80]:
df[[
    "disbursement_date",
    "loan_year",
    "loan_month",
    "loan_day_of_week"
]].head()

,disbursement_date,loan_year,loan_month,loan_day_of_week
0,2022-08-15,2022,8,0
1,2022-07-13,2022,7,2
2,2022-09-12,2022,9,0
3,2022-09-29,2022,9,3
4,2022-08-27,2022,8,5


In [81]:
df[["loan_year", "loan_month", "loan_day_of_week"]].nunique()

loan_year            4
loan_month          12
loan_day_of_week     7
dtype: int64

#### 2. Financial Features

In [82]:
# Interest amount

df["interest_amount"] = (
    df["total_amount_to_repay"] - df["total_amount"]
)

In [83]:
df[[
    "total_amount",
    "total_amount_to_repay",
    "interest_amount"
]].head()

,total_amount,total_amount_to_repay,interest_amount
0,16477.0,16642.0,165.0
1,5599.0,5599.0,0.0
2,4679.0,4746.0,67.0
3,6123.0,6123.0,0.0
4,2953.0,2974.0,21.0


In [84]:
# repayment ratio

df["repayment_ratio"] = (
    df["total_amount_to_repay"] / df["total_amount"]
)

In [85]:
df[[
    "total_amount",
    "total_amount_to_repay",
    "repayment_ratio"
]].head()

,total_amount,total_amount_to_repay,repayment_ratio
0,16477.0,16642.0,1.010014
1,5599.0,5599.0,1.000000
2,4679.0,4746.0,1.014319
3,6123.0,6123.0,1.000000
4,2953.0,2974.0,1.007111


In [86]:

df["repayment_ratio"].describe()

count    68654.000000
mean         1.021686
std          0.055755
min          0.000000
25%          1.000000
50%          1.008210
75%          1.030400
max          3.164865
Name: repayment_ratio, dtype: float64

In [87]:
df.loc[
    df["repayment_ratio"] == 0,
    [
        "total_amount",
        "total_amount_to_repay",
        "interest_amount",
        "repayment_ratio",
        "target"
    ]
].head(10)

,total_amount,total_amount_to_repay,interest_amount,repayment_ratio,target
2389,8550.0,0.0,-8550.0,0.0,0
3477,500.0,0.0,-500.0,0.0,0
50390,2259.0,0.0,-2259.0,0.0,0
52937,290.0,0.0,-290.0,0.0,0


In [88]:
df.loc[
    df["repayment_ratio"] > 2,
    [
        "total_amount",
        "total_amount_to_repay",
        "interest_amount",
        "repayment_ratio",
        "target"
    ]
].head(10)

,total_amount,total_amount_to_repay,interest_amount,repayment_ratio,target
62232,6887.88,15977.38,9089.5,2.319637,1
65701,210.00,458.00,248.0,2.180952,1
65903,740.00,2342.00,1602.0,3.164865,1


In [89]:
df["repayment_ratio_zero"] = (
    df["total_amount_to_repay"] == 0
).astype(int)

#### 3. Customer-Based Features

In [90]:
df = df.sort_values(
    ["customer_id", "disbursement_date"]
).reset_index(drop=True)

In [91]:
df[
    [
        "customer_id",
        "disbursement_date",
        "target"
    ]
].head(10)

,customer_id,disbursement_date,target
0,145,2022-03-09,0
1,161,2024-01-15,0
2,166,2022-03-11,0
3,1487,2022-03-14,0
4,4534,2022-03-07,0
5,4534,2023-07-13,0
6,6083,2022-03-21,0
7,6083,2022-04-13,0
8,6083,2022-05-11,0
9,6083,2022-06-08,0


In [92]:
customer_date = (
    df.groupby(["customer_id", "disbursement_date"])
      .agg(
          loan_records=("id", "count"),
          default_records=("target", "sum")
      )
      .reset_index()
)

customer_date = customer_date.sort_values(
    ["customer_id", "disbursement_date"]
).reset_index(drop=True)

customer_date.head()

,customer_id,disbursement_date,loan_records,default_records
0,145,2022-03-09,1,0
1,161,2024-01-15,1,0
2,166,2022-03-11,1,0
3,1487,2022-03-14,1,0
4,4534,2022-03-07,1,0


In [93]:
customer_date["previous_loan_count"] = (
    customer_date.groupby("customer_id")["loan_records"]
    .transform(lambda x: x.cumsum().shift(fill_value=0))
)

customer_date["previous_default_count"] = (
    customer_date.groupby("customer_id")["default_records"]
    .transform(lambda x: x.cumsum().shift(fill_value=0))
)

customer_date.head(10)

,customer_id,disbursement_date,loan_records,default_records,previous_loan_count,previous_default_count
0,145,2022-03-09,1,0,0,0
1,161,2024-01-15,1,0,0,0
2,166,2022-03-11,1,0,0,0
3,1487,2022-03-14,1,0,0,0
4,4534,2022-03-07,1,0,0,0
5,4534,2023-07-13,1,0,1,0
6,6083,2022-03-21,1,0,0,0
7,6083,2022-04-13,1,0,1,0
8,6083,2022-05-11,1,0,2,0
9,6083,2022-06-08,1,0,3,0


In [94]:
history_features = customer_date[
    [
        "customer_id",
        "disbursement_date",
        "previous_loan_count",
        "previous_default_count"
    ]
]

df = df.drop(
    columns=["previous_loan_count", "previous_default_count"],
    errors="ignore"
)

df = df.merge(
    history_features,
    on=["customer_id", "disbursement_date"],
    how="left",
    validate="many_to_one"
)

df.shape

(68654, 24)

In [95]:
df["previous_default_rate"] = np.where(
    df["previous_loan_count"] > 0,
    df["previous_default_count"] / df["previous_loan_count"],
    0
)

In [96]:
df[
    [
        "customer_id",
        "disbursement_date",
        "previous_loan_count",
        "previous_default_count",
        "previous_default_rate"
    ]
].head(10)

,customer_id,disbursement_date,previous_loan_count,previous_default_count,previous_default_rate
0,145,2022-03-09,0,0,0.0
1,161,2024-01-15,0,0,0.0
2,166,2022-03-11,0,0,0.0
3,1487,2022-03-14,0,0,0.0
4,4534,2022-03-07,0,0,0.0
5,4534,2023-07-13,1,0,0.0
6,6083,2022-03-21,0,0,0.0
7,6083,2022-04-13,1,0,0.0
8,6083,2022-05-11,2,0,0.0
9,6083,2022-06-08,3,0,0.0


In [97]:
df["previous_default_rate"].describe()

count    68654.000000
mean         0.001125
std          0.028836
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: previous_default_rate, dtype: float64

In [98]:
df["previous_default_count"].value_counts().sort_index()

previous_default_count
0     68489
1        79
2        46
3        19
4         6
5         2
6         2
7         4
11        2
13        2
15        3
Name: count, dtype: int64

In [99]:
customer_date["previous_disbursement_date"] = (
    customer_date.groupby("customer_id")["disbursement_date"]
    .shift(1)
)

customer_date["days_since_previous_loan"] = (
    customer_date["disbursement_date"]
    - customer_date["previous_disbursement_date"]
).dt.days

In [100]:
customer_date[
    [
        "customer_id",
        "disbursement_date",
        "previous_disbursement_date",
        "days_since_previous_loan"
    ]
].head(10)

,customer_id,disbursement_date,previous_disbursement_date,days_since_previous_loan
0,145,2022-03-09,NaT,NaN
1,161,2024-01-15,NaT,NaN
2,166,2022-03-11,NaT,NaN
3,1487,2022-03-14,NaT,NaN
4,4534,2022-03-07,NaT,NaN
5,4534,2023-07-13,2022-03-07,493.0
6,6083,2022-03-21,NaT,NaN
7,6083,2022-04-13,2022-03-21,23.0
8,6083,2022-05-11,2022-04-13,28.0
9,6083,2022-06-08,2022-05-11,28.0


In [101]:
history_features = customer_date[
    [
        "customer_id",
        "disbursement_date",
        "days_since_previous_loan"
    ]
]

df = df.drop(
    columns=["days_since_previous_loan"],
    errors="ignore"
)

df = df.merge(
    history_features,
    on=["customer_id", "disbursement_date"],
    how="left",
    validate="many_to_one"
)

df.shape

(68654, 26)

In [102]:
df["first_loan"] = (
    df["days_since_previous_loan"].isna()
).astype(int)

df["days_since_previous_loan"] = (
    df["days_since_previous_loan"].fillna(0)
)

In [103]:
df[
    [
        "first_loan",
        "days_since_previous_loan"
    ]
].describe()

,first_loan,days_since_previous_loan
count,68654.000000,68654.000000
mean,0.106199,8.677105
std,0.308095,23.149113
min,0.000000,0.000000
25%,0.000000,2.000000
50%,0.000000,5.000000
75%,0.000000,9.000000
max,1.000000,787.000000


In [104]:
economic_df[["Country", "Indicator"]]

,Country,Indicator
0,Ghana,"Inflation, consumer prices (annual %)"
1,Cote d'Ivoire,"Inflation, consumer prices (annual %)"
2,Kenya,"Inflation, consumer prices (annual %)"
3,Ghana,"Official exchange rate (LCU per US$, period av..."
4,Cote d'Ivoire,"Official exchange rate (LCU per US$, period av..."
5,Kenya,"Official exchange rate (LCU per US$, period av..."
6,Ghana,Real interest rate (%)
7,Cote d'Ivoire,Real interest rate (%)
8,Kenya,Real interest rate (%)
9,Ghana,Average precipitation in depth (mm per year)


In [105]:
kenya_economic = economic_df[
    economic_df["Country"] == "Kenya"
].copy()

kenya_economic.isna().sum()

Country      0
Indicator    0
YR2001       0
YR2002       0
YR2003       0
YR2004       0
YR2005       0
YR2006       0
YR2007       0
YR2008       0
YR2009       0
YR2010       0
YR2011       0
YR2012       0
YR2013       0
YR2014       0
YR2015       1
YR2016       1
YR2017       1
YR2018       1
YR2019       1
YR2020       1
YR2021       1
YR2022       2
YR2023       2
dtype: int64

In [106]:
kenya_economic.shape

(9, 25)

In [107]:
kenya_economic.set_index("Indicator").loc[:, "YR2015":"YR2023"].isna()

,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023
Indicator,,,,,,,,,
"Inflation, consumer prices (annual %)",False,False,False,False,False,False,False,False,False
"Official exchange rate (LCU per US$, period average)",False,False,False,False,False,False,False,False,False
Real interest rate (%),False,False,False,False,False,False,False,False,False
Average precipitation in depth (mm per year),False,False,False,False,False,False,False,True,True
Deposit interest rate (%),False,False,False,False,False,False,False,False,False
Lending interest rate (%),False,False,False,False,False,False,False,False,False
"Interest rate spread (lending rate minus deposit rate, %)",False,False,False,False,False,False,False,False,False
Fossil fuel energy consumption (% of total),True,True,True,True,True,True,True,True,True
Unemployment rate,False,False,False,False,False,False,False,False,False


In [108]:
kenya_economic = kenya_economic[
    ~kenya_economic["Indicator"].isin([
        "Average precipitation in depth (mm per year)",
        "Fossil fuel energy consumption (% of total)"
    ])
].copy()

kenya_economic.shape

(7, 25)

In [109]:
economic_long = kenya_economic.melt(
    id_vars=["Country", "Indicator"],
    var_name="year",
    value_name="value"
)

economic_long.head()

,Country,Indicator,year,value
0,Kenya,"Inflation, consumer prices (annual %)",YR2001,5.738598
1,Kenya,"Official exchange rate (LCU per US$, period av...",YR2001,78.563195
2,Kenya,Real interest rate (%),YR2001,17.812501
3,Kenya,Deposit interest rate (%),YR2001,6.639167
4,Kenya,Lending interest rate (%),YR2001,19.665833


In [110]:
economic_long["year"] = (
    economic_long["year"]
    .str.replace("YR", "")
    .astype(int)
)

economic_wide = (
    economic_long
    .pivot(
        index="year",
        columns="Indicator",
        values="value"
    )
    .reset_index()
)

economic_wide.head()

Indicator,year,Deposit interest rate (%),"Inflation, consumer prices (annual %)","Interest rate spread (lending rate minus deposit rate, %)",Lending interest rate (%),"Official exchange rate (LCU per US$, period average)",Real interest rate (%),Unemployment rate
0,2001,6.639167,5.738598,13.026667,19.665833,78.563195,17.812501,2.868
1,2002,5.486667,1.961308,12.966667,18.453333,78.749142,17.358141,2.903
2,2003,4.133333,9.815691,12.440000,16.573333,75.935569,9.770511,2.860
3,2004,2.433333,11.624036,10.098333,12.531667,79.173876,5.045258,2.836
4,2005,5.082500,10.312778,7.800000,12.882500,75.554109,7.609988,2.739


In [111]:
economic_wide = economic_wide.rename(columns={
    "Deposit interest rate (%)": "deposit_interest_rate",
    "Inflation, consumer prices (annual %)": "inflation_rate",
    "Interest rate spread (lending rate minus deposit rate, %)": "interest_rate_spread",
    "Lending interest rate (%)": "lending_interest_rate",
    "Official exchange rate (LCU per US$, period average)": "exchange_rate",
    "Real interest rate (%)": "real_interest_rate",
    "Unemployment rate": "unemployment_rate"
})

economic_wide.head()

Indicator,year,deposit_interest_rate,inflation_rate,interest_rate_spread,lending_interest_rate,exchange_rate,real_interest_rate,unemployment_rate
0,2001,6.639167,5.738598,13.026667,19.665833,78.563195,17.812501,2.868
1,2002,5.486667,1.961308,12.966667,18.453333,78.749142,17.358141,2.903
2,2003,4.133333,9.815691,12.440000,16.573333,75.935569,9.770511,2.860
3,2004,2.433333,11.624036,10.098333,12.531667,79.173876,5.045258,2.836
4,2005,5.082500,10.312778,7.800000,12.882500,75.554109,7.609988,2.739


In [112]:
df.columns.tolist()

['id',
 'customer_id',
 'country_id',
 'tbl_loan_id',
 'lender_id',
 'loan_type',
 'total_amount',
 'total_amount_to_repay',
 'disbursement_date',
 'due_date',
 'duration',
 'new_versus_repeat',
 'amount_funded_by_lender',
 'lender_portion_funded',
 'lender_portion_to_be_repaid',
 'target',
 'loan_year',
 'loan_month',
 'loan_day_of_week',
 'interest_amount',
 'repayment_ratio',
 'repayment_ratio_zero',
 'previous_loan_count',
 'previous_default_count',
 'previous_default_rate',
 'days_since_previous_loan',
 'first_loan']

In [113]:
df = df.merge(
    economic_wide,
    left_on="loan_year",
    right_on="year",
    how="left",
    validate="many_to_one"
).drop(columns="year")

df.shape

(68654, 34)

In [114]:
df[
    [
        "inflation_rate",
        "lending_interest_rate",
        "exchange_rate",
        "unemployment_rate"
    ]
].describe()

,inflation_rate,lending_interest_rate,exchange_rate,unemployment_rate
count,65725.000000,65725.000000,65725.000000,65725.000000
mean,7.659199,12.358511,118.261669,5.802697
std,0.036849,0.167634,2.946304,0.016652
min,6.107936,12.079998,109.637747,5.682000
25%,7.659863,12.335841,117.865989,5.805000
50%,7.659863,12.335841,117.865989,5.805000
75%,7.659863,12.335841,117.865989,5.805000
max,7.671396,13.588502,139.846384,5.805000


In [115]:
df.loc[
    df["inflation_rate"].isna(),
    "loan_year"
].value_counts().sort_index()

loan_year
2024    2929
Name: count, dtype: int64

In [116]:
df[
    [
        "deposit_interest_rate",
        "inflation_rate",
        "interest_rate_spread",
        "lending_interest_rate",
        "exchange_rate",
        "real_interest_rate",
        "unemployment_rate"
    ]
].isna().sum()

deposit_interest_rate    2929
inflation_rate           2929
interest_rate_spread     2929
lending_interest_rate    2929
exchange_rate            2929
real_interest_rate       2929
unemployment_rate        2929
dtype: int64

In [117]:
df["economic_data_missing"] = (
    df["inflation_rate"].isna()
).astype(int)

df["economic_data_missing"].value_counts()

economic_data_missing
0    65725
1     2929
Name: count, dtype: int64